# Submit SCF energy calculation

In [ ]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

In [ ]:
# General imports.
import urllib.parse as urlparse

import ipywidgets as ipw
from IPython.display import clear_output

# AiiDA imports.
%load_ext aiida
%aiida
# AiiDAlab imports.
import aiidalab_widgets_base as awb
import aiidalab_widgets_empa as awe
from aiida import orm, plugins

from surfaces_tools.utils import wfn

# Custom imports.
from surfaces_tools.widgets import editors, inputs

In [ ]:
Cp2kScfWorkChain = plugins.WorkflowFactory("nanotech_empa.cp2k.scf")

In [ ]:
# Structure selector.
build_slab = editors.BuildSlab(title="Build slab")
input_details = inputs.InputDetails()

structure_selector = awb.StructureManagerWidget(
    importers=[
        awb.StructureUploadWidget(title="Import from computer"),
        awb.StructureBrowserWidget(title="AiiDA database"),
        awb.SmilesWidget(title="From SMILES"),
        awe.CdxmlUploadWidget(title="CDXML"),
    ],
    editors=[
        awb.BasicStructureEditor(title="Edit structure"),
        build_slab,
        awb.BasicCellEditor(),
        editors.InsertStructureWidget(title="Insert molecule"),
    ],
    storable=True,
    node_class="StructureData",
)
ipw.dlink((structure_selector, "structure"), (build_slab, "molecule"))
ipw.dlink((structure_selector, "structure"), (input_details, "structure"))
ipw.dlink((structure_selector, "structure_node"), (input_details, "structure_node"))
ipw.dlink((input_details, "details"), (build_slab, "details"))
display(structure_selector)

# Code.
code_input_widget = awb.ComputationalResourcesWidget(
    description="CP2K code:", default_calc_job_plugin="cp2k"
)
sparse_overlap_code = awb.ComputationalResourcesWidget(
    description="Overlap converter:",
    default_calc_job_plugin="nanotech_empa.sparse_overlap",
)
resources = awe.ProcessResourcesWidget()

output = ipw.Output()

In [ ]:
url = urlparse.urlsplit(jupyter_notebook_url)
parsed_url = urlparse.parse_qs(url.query)
if "structure_uuid" in parsed_url:
    structure_selector.input_structure = load_node(parsed_url["structure_uuid"][0])

In [ ]:
# Protocol.
protocol = ipw.Dropdown(
    value="standard",
    options=[
        ("Standard", "standard"),
        ("Low accuracy", "low_accuracy"),
        ("Debug", "debug"),
    ],
    description="Protocol:",
    style={"description_width": "initial"},
)

write_overlap_matrix = ipw.Checkbox(
    value=False,
    description="Write overlap matrix",
    style={"description_width": "initial"},
)

diagonalisation_smearing = inputs.DiagonalisationSmearingWidget()

added_mos = ipw.BoundedIntText(
    value=100,
    min=0,
    max=100000,
    description="ADDED_MOS:",
    style={"description_width": "initial"},
    layout={"width": "220px"},
)

overlap_ndigits = ipw.BoundedIntText(
    value=14,
    min=1,
    max=30,
    description="Overlap digits:",
    style={"description_width": "initial"},
    layout={"width": "220px"},
)

retrieve_sparse_overlap = ipw.Checkbox(
    value=False,
    description="Retrieve sparse overlap matrix",
    style={"description_width": "initial"},
)

overlap_threshold = ipw.FloatText(
    value=1.0e-10,
    description="Sparse threshold:",
    style={"description_width": "initial"},
    layout={"width": "240px"},
)

sparse_retrieve_options = ipw.VBox([])
overlap_options = ipw.VBox([])


def _update_sparse_retrieve_options(_=None):
    sparse_retrieve_options.children = (
        [overlap_threshold, sparse_overlap_code]
        if retrieve_sparse_overlap.value
        else []
    )


def _update_overlap_options(_=None):
    overlap_options.children = (
        [
            diagonalisation_smearing,
            added_mos,
            overlap_ndigits,
            retrieve_sparse_overlap,
            sparse_retrieve_options,
        ]
        if write_overlap_matrix.value
        else []
    )
    _update_sparse_retrieve_options()

write_overlap_matrix.observe(_update_overlap_options, "value")
retrieve_sparse_overlap.observe(_update_sparse_retrieve_options, "value")
_update_overlap_options()

parent_calc_folder = ipw.Text(
    value="",
    description="WFN restart folder:",
    placeholder="Optional remote path on the selected computer",
    style={"description_width": "initial"},
    layout={"width": "70%"},
)

advanced_settings = ipw.Accordion(
    children=[ipw.VBox([parent_calc_folder])],
    selected_index=None,
)
advanced_settings.set_title(0, "Advanced settings")

In [ ]:
workflow_description = ipw.Text(
    description="Workflow description:",
    placeholder="Provide the description here.",
    style={"description_width": "initial"},
    layout={"width": "70%"},
)

In [ ]:
ipw.dlink((code_input_widget, "value"), (input_details, "selected_code"))


def prepare_scf():
    with output:
        clear_output()
    if not structure_selector.structure_node:
        can_submit, msg = False, "Select a structure first."
    elif not code_input_widget.value:
        can_submit, msg = False, "Select CP2K code."
    elif write_overlap_matrix.value and retrieve_sparse_overlap.value and not sparse_overlap_code.value:
        can_submit, msg = False, "Select overlap converter code."
    else:
        can_submit, msg, parameters = input_details.return_final_dictionary()
        parent_calc_folder_path = parent_calc_folder.value.strip()

    if not can_submit:
        with output:
            print(msg)
            return

    code = orm.load_code(code_input_widget.value)
    dft_params_dict = parameters["dft_params"]
    dft_params_dict.setdefault("periodic", "XYZ")
    dft_params_dict["elpa_switch"] = True
    dft_params_dict["sc_diag"] = diagonalisation_smearing.enable_diagonalisation.value

    if write_overlap_matrix.value and added_mos.value > 0:
        dft_params_dict["added_mos"] = added_mos.value

    if write_overlap_matrix.value and diagonalisation_smearing.smearing_enabled:
        dft_params_dict["smear_t"] = diagonalisation_smearing.smearing_temperature.value
        dft_params_dict["force_multiplicity"] = diagonalisation_smearing.force_multiplicity.value

    builder = Cp2kScfWorkChain.get_builder()
    builder.protocol = orm.Str(protocol.value)
    builder.metadata.label = (
        "CP2K_SCF_overlap" if write_overlap_matrix.value else "CP2K_SCF"
    )
    builder.metadata.description = workflow_description.value
    builder.cp2k_code = code
    builder.options = {
        "max_wallclock_seconds": resources.walltime_seconds,
        "resources": {
            "num_machines": resources.nodes,
            "num_mpiprocs_per_machine": resources.tasks_per_node,
            "num_cores_per_mpiproc": resources.threads_per_task,
        },
    }
    builder.structure = structure_selector.structure_node
    builder.dft_params = orm.Dict(dft_params_dict)
    builder.write_overlap_matrix = orm.Bool(write_overlap_matrix.value)
    builder.retrieve_sparse_overlap = orm.Bool(retrieve_sparse_overlap.value)
    builder.overlap_ndigits = orm.Int(overlap_ndigits.value)
    builder.overlap_threshold = orm.Float(overlap_threshold.value)
    if retrieve_sparse_overlap.value:
        builder.sparse_overlap_code = orm.load_code(sparse_overlap_code.value)

    if parent_calc_folder_path:
        selected_parent_calc_folder = orm.RemoteData(
            computer=code.computer,
            remote_path=parent_calc_folder_path,
        )
        print(
            "Restarting from selected parent folder: "
            f"{parent_calc_folder_path}"
        )
        builder.parent_calc_folder = selected_parent_calc_folder
    else:
        # Check if a restart wfn is available.
        wave_function = None
        if structure_selector.structure_node.is_stored:
            wave_function = wfn.structure_available_wfn(
                node=structure_selector.structure_node,
                relative_replica_id=None,
                current_hostname=code.computer.hostname,
                return_path=False,
                dft_params=dft_params_dict,
            )
        if wave_function is not None:
            print(f"Restarting from wfn in folder: {wave_function.pk}")
            builder.parent_calc_folder = wave_function

    return builder

In [ ]:
btn_submit = awb.SubmitButtonWidget(
    Cp2kScfWorkChain,
    inputs_generator=prepare_scf,
    disable_after_submit=False,
    append_output=True,
)

In [ ]:
# Resources estimation.
resources_estimation = awe.ResourcesEstimatorWidget(
    price_link="https://2go.cscs.ch/offering/swiss_academia/institutional_customers/",
    price_per_hour=2.85,
)
resources_estimation.link_to_resources_widget(resources)
ipw.dlink((input_details, "details"), (resources_estimation, "details"))
ipw.dlink((input_details, "uks"), (resources_estimation, "uks"))
_ = ipw.dlink((code_input_widget, "value"), (resources_estimation, "selected_code"))

# Inputs

In [ ]:
display(input_details, protocol, write_overlap_matrix, overlap_options, advanced_settings)

# Code and resources

In [ ]:
display(code_input_widget, ipw.HBox([resources, resources_estimation]))

# Submit

In [ ]:
display(workflow_description, btn_submit, output)